<a href="https://colab.research.google.com/github/saiqadev/pythoncodes/blob/main/latex2docx.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# LaTeX to DOCX Conversion in Google Colab
# Complete Notebook Template

# =============================================================================
# CELL 1: Install dependencies
# =============================================================================

!apt-get update -qq
!apt-get install -y -qq pandoc
!apt-get install -y -qq texlive-latex-base texlive-fonts-recommended texlive-fonts-extra

print("✓ Dependencies installed")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Preconfiguring packages ...
Selecting previously unselected package fonts-droid-fallback.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../00-fonts-droid-fallback_1%3a6.0.1r16-1.1build1_all.deb ...
Unpacking fonts-droid-fallback (1:6.0.1r16-1.1build1) ...
Selecting previously unselected package fonts-lato.
Preparing to unpack .../01-fonts-lato_2.0-2.1_all.deb ...
Unpacking fonts-lato (2.0-2.1) ...
Selecting previously unselected package poppler-data.
Preparing to unpack .../02-poppler-data_0.4.11-1_all.deb ...
Unpacking poppler-data (0.4.11-1) ...
Selecting previously unselected package tex-common.
Preparing to unpack .../03-tex-common_6.17_all.deb ...
Unpacking tex-common (6.17) ...
Selecting previously unselect

In [2]:
# =============================================================================
# CELL 2: Import and setup
# =============================================================================

import os
import sys
import re
import json
import shutil
import zipfile
import subprocess
from pathlib import Path
from typing import Dict, List, Optional
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("✓ Imports successful")


✓ Imports successful


In [3]:
# =============================================================================
# CELL 3: Define converter class (paste the LatexToDocxConverter class)
# =============================================================================

# [Paste the LatexToDocxConverter class from latex_to_docx_converter.py here]
# For brevity, we'll create a simplified version inline

from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

@dataclass
class ConversionConfig:
    """Configuration for LaTeX to DOCX conversion"""
    input_zip: str
    output_docx: str
    main_file: str = "main.tex"
    extract_dir: Optional[str] = None
    use_pandoc: bool = True
    preserve_images: bool = True
    generate_toc: bool = True


class LatexToDocxConverter:
    """Convert LaTeX/Overleaf projects to DOCX format"""

    def __init__(self, config: ConversionConfig):
        self.config = config
        self.extract_dir = Path(config.extract_dir or "./latex_project")
        self.images_dir = self.extract_dir / "images"
        self.bibtex_files = []

    def extract_zip(self):
        """Extract the zipped LaTeX project"""
        logger.info(f"Extracting {self.config.input_zip}...")

        if self.extract_dir.exists():
            shutil.rmtree(self.extract_dir)

        self.extract_dir.mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(self.config.input_zip, 'r') as zip_ref:
            zip_ref.extractall(self.extract_dir)

        # Handle Overleaf wrapper directory
        contents = list(self.extract_dir.iterdir())
        if len(contents) == 1 and contents[0].is_dir():
            wrapper = contents[0]
            for item in wrapper.iterdir():
                shutil.move(str(item), str(self.extract_dir / item.name))
            wrapper.rmdir()

        logger.info(f"✓ Extracted to {self.extract_dir}")

    def find_main_tex(self) -> Path:
        """Find the main .tex file"""
        main = self.extract_dir / self.config.main_file
        if main.exists():
            return main

        for name in ["main.tex", "document.tex", "thesis.tex", "paper.tex"]:
            candidate = self.extract_dir / name
            if candidate.exists():
                logger.info(f"✓ Found main file: {candidate}")
                return candidate

        for tex_file in self.extract_dir.glob("*.tex"):
            content = tex_file.read_text(errors='ignore')
            if "\\documentclass" in content:
                logger.info(f"✓ Found main file: {tex_file}")
                return tex_file

        raise FileNotFoundError("Could not find main LaTeX file")

    def find_bibtex_files(self) -> List[Path]:
        """Find all BibTeX files"""
        bibtex_files = []
        for bib_file in self.extract_dir.glob("**/*.bib"):
            bibtex_files.append(bib_file)
            logger.info(f"✓ Found BibTeX file: {bib_file}")
        return bibtex_files

    def find_images(self) -> Dict[str, Path]:
        """Find all image files"""
        image_extensions = {'.png', '.jpg', '.jpeg', '.pdf', '.eps', '.svg'}
        images = {}
        for img_file in self.extract_dir.glob("**/*"):
            if img_file.suffix.lower() in image_extensions:
                images[img_file.name] = img_file
                logger.info(f"✓ Found image: {img_file.name}")
        return images

    def preprocess_latex(self, tex_content: str) -> str:
        """Preprocess LaTeX for better conversion"""
        # Remove comments
        tex_content = re.sub(r'%.*?$', '', tex_content, flags=re.MULTILINE)
        # Process includes
        tex_content = self._process_includes(tex_content)
        return tex_content

    def _process_includes(self, tex_content: str) -> str:
        """Handle \input and \include commands"""
        def replace_include(match):
            cmd = match.group(1)
            file_path = match.group(2).strip()

            search_paths = [
                self.extract_dir / f"{file_path}.tex",
                self.extract_dir / file_path,
                self.extract_dir / "sections" / f"{file_path}.tex",
                self.extract_dir / "chapters" / f"{file_path}.tex",
            ]

            for path in search_paths:
                if path.exists():
                    try:
                        included_content = path.read_text(encoding='utf-8', errors='ignore')
                        logger.info(f"✓ Including: {path}")
                        return self._process_includes(included_content)
                    except Exception as e:
                        logger.warning(f"Failed to include {path}: {e}")

            logger.warning(f"Could not find: {file_path}")
            return match.group(0)

        tex_content = re.sub(r'\\(input|include)\s*\{([^}]+)\}', replace_include, tex_content)
        return tex_content

    def convert_with_pandoc(self, tex_file: Path) -> bool:
        """Convert using Pandoc"""
        try:
            logger.info(f"Converting {tex_file} to DOCX...")

            tex_content = tex_file.read_text(encoding='utf-8', errors='ignore')
            tex_content = self.preprocess_latex(tex_content)

            temp_tex = self.extract_dir / "_temp_processed.tex"
            temp_tex.write_text(tex_content, encoding='utf-8')

            cmd = [
                "pandoc",
                str(temp_tex),
                "-f", "latex",
                "-t", "docx",
                "-o", str(self.config.output_docx),
                "--standalone",
                "--toc" if self.config.generate_toc else "",
                "--number-sections",
                "--metadata", "title=LaTeX Document",
            ]

            cmd = [c for c in cmd if c]

            if self.images_dir.exists():
                cmd.extend(["--resource-path", str(self.extract_dir)])

            result = subprocess.run(cmd, capture_output=True, text=True)

            if result.returncode != 0:
                logger.error(f"Pandoc error: {result.stderr}")
                return False

            logger.info(f"✓ Created: {self.config.output_docx}")

            if temp_tex.exists():
                temp_tex.unlink()

            return True

        except Exception as e:
            logger.error(f"Conversion error: {e}")
            return False

    def convert(self) -> bool:
        """Main conversion"""
        try:
            logger.info("Starting conversion...")

            self.extract_zip()
            main_tex = self.find_main_tex()
            self.bibtex_files = self.find_bibtex_files()
            images = self.find_images()

            logger.info(f"Project summary: {len(images)} images, {len(self.bibtex_files)} BibTeX files")

            success = self.convert_with_pandoc(main_tex)

            if success:
                logger.info("✓ Conversion completed successfully!")
                return True
            else:
                logger.error("✗ Conversion failed")
                return False

        except Exception as e:
            logger.error(f"Conversion error: {e}")
            import traceback
            traceback.print_exc()
            return False


print("✓ Converter class defined")




✓ Converter class defined


<>:101: SyntaxWarning: invalid escape sequence '\i'
<>:101: SyntaxWarning: invalid escape sequence '\i'
/tmp/ipykernel_11863/3772313922.py:101: SyntaxWarning: invalid escape sequence '\i'
  """Handle \input and \include commands"""


In [9]:
# =============================================================================
# CELL 4: Upload your LaTeX zip file
# =============================================================================

from google.colab import files

print("Upload your LaTeX/Overleaf zip file:")
uploaded = files.upload()

zip_file = list(uploaded.keys())[0] if uploaded else None
if zip_file:
    print(f"✓ Uploaded: {zip_file}")
else:
    print("✗ No file uploaded")




Upload your LaTeX/Overleaf zip file:


Saving ZippedLSR_Template.zip to ZippedLSR_Template.zip
✓ Uploaded: ZippedLSR_Template.zip


In [10]:

# =============================================================================
# CELL 5: Configure and run conversion
# =============================================================================

# Configuration
OUTPUT_DOCX = "output.docx"
MAIN_TEX = "main.tex"  # Change if your main file has a different name
EXTRACT_DIR = "./latex_project"

# Create config
config = ConversionConfig(
    input_zip=zip_file,
    output_docx=OUTPUT_DOCX,
    main_file=MAIN_TEX,
    extract_dir=EXTRACT_DIR,
    use_pandoc=True,
    preserve_images=True,
    generate_toc=True
)

# Run conversion
converter = LatexToDocxConverter(config)
success = converter.convert()

if success:
    print(f"\n✓ Conversion successful!")
    print(f"Output file: {OUTPUT_DOCX}")
else:
    print("\n✗ Conversion failed - check logs above")





✓ Conversion successful!
Output file: output.docx


In [11]:

# =============================================================================
# CELL 6: Download the converted DOCX file
# =============================================================================

if success and Path(OUTPUT_DOCX).exists():
    from google.colab import files
    print("Downloading your DOCX file...")
    files.download(OUTPUT_DOCX)
    print("✓ Download started!")
else:
    print("✗ Output file not found")



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Download started!


In [8]:

# =============================================================================
# CELL 7 (Optional): Verify output
# =============================================================================

if Path(OUTPUT_DOCX).exists():
    file_size_mb = Path(OUTPUT_DOCX).stat().st_size / (1024 * 1024)
    print(f"Output file size: {file_size_mb:.2f} MB")

    # List extracted files for inspection
    extract_path = Path(EXTRACT_DIR)
    if extract_path.exists():
        print("\nExtracted project structure:")
        for item in sorted(extract_path.glob("*"))[:20]:  # Show first 20 items
            item_type = "[DIR]" if item.is_dir() else "[FILE]"
            print(f"  {item_type} {item.name}")
